Copyright 2026 Google LLC.

SPDX-License-Identifier: Apache-2.0


<a href="https://colab.research.google.com/github/datacommonsorg/api-python/blob/master/notebooks/analyzing_census_data_sdmx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Analyzing Census Data with Data Commons SDMX

Every year, the American Community Survey (published by the US Census) reports thousands of variables about demographics, economics, housing, and more. This information, stored in Data Commons, is available to everyone for data science projects, education, and exploration. This tutorial introduces the Data Commons graph and two of its tools to help integrate its data into data science projects: the knowledge graph [browser](https://datacommons.org/browser) and the SDMX APIs used from Python.


## What is Data Commons?

Data Commons is an open knowledge repository that combines data from public datasets using mapped common entities. It contains statements about real-world objects such as:

- [Santa Clara County](https://datacommons.org/browser/geoId/06085) is contained in the [State of California](https://datacommons.org/browser/geoId/06).
- The latitude of [Berkeley, CA](https://datacommons.org/browser/geoId/0606000) is 37.8703.
- [The population of Maryland](https://datacommons.org/browser/geoId/24) was [6.26 M in 2024](https://datacommons.org/browser/geoId/24?statVar=Count_Person).

In the graph, [entities](https://docs.datacommons.org/data_model.html#entity) like Santa Clara County are represented by nodes. Every node has a type corresponding to what the node represents. For example, California is a [State](https://datacommons.org/browser/State). Relations between entities are represented by edges between these nodes. The statement "Santa Clara County is contained in the State of California" is represented by two nodes connected by an edge labeled [containedInPlace](https://datacommons.org/browser/containedInPlace). Data Commons closely follows the [Schema.org data model](https://schema.org/docs/datamodel.html) and uses Schema.org schema to provide a common set of types and properties.


## Data Commons knowledge graph browser

The [Data Commons browser](https://datacommons.org/browser) provides a way to explore the data in a human-readable format. It is the best way to explore what is in Data Commons. Searching in the browser for an entity like [Mountain View](https://datacommons.org/browser/geoId/0649670) takes you to a page about the entity, including properties such as [containedInPlace](https://datacommons.org/browser/containedInPlace) and [timezone](https://datacommons.org/browser/timezone).

An important property for all entities is the **DCID** (Data Commons identifier). This is a unique identifier assigned to each entity in the knowledge graph. The DCID is listed at the top of a browser page next to "About:" and in the list of properties.


## SDMX APIs from Python

The Data Commons SDMX APIs provide statistical observations in a standard format. This notebook uses:

- **SDMX Availability** to confirm that the requested statistical variables are available for a selected geography and source facet.
- **SDMX Data** to retrieve observation values as SDMX-CSV.
- **The Data Commons Python client** to retrieve human-readable names for the place DCIDs returned by SDMX.

pySDMX reads Availability responses. Pandas reads the SDMX-CSV data into DataFrames, which can be used with data-processing, analytical, and visualization packages such as Pandas, NumPy, SciPy, and Matplotlib.

## Set up environment

### Install libraries

Install pySDMX for SDMX metadata and the Data Commons Python client for place names.


In [ ]:
!pip install "pysdmx[data,json]==1.19.0" "datacommons-client==2.1.6" --quiet


### Configure the API key

Obtain your own Data Commons API key from [apikeys.datacommons.org](https://apikeys.datacommons.org/) and replace the placeholder in the cell below with your key.

Set the key once below. The notebook sends it in the `X-Api-Key` header for SDMX requests and configures the Data Commons Python client with the same key. After changing the key, rerun this cell and the data cells below; the installation, imports, and helper definitions do not need to be rerun.


In [ ]:
from datacommons_client import DataCommonsClient

DC_API_KEY = "Replace this string with your API key"
dc_client = DataCommonsClient(api_key=DC_API_KEY)


### Import dependencies

Import the libraries and define the helpers required for data manipulation, analysis, and plotting.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import io
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from pysdmx.io import read_sdmx

API_ROOT = "https://api.datacommons.org"
SDMX_DATA_URL = f"{API_ROOT}/sdmx/v3/data/dataflow/DC/DF_OBS/1.0.0/*"
SDMX_AVAILABILITY_URL = (
    f"{API_ROOT}/sdmx/v3/availability/dataflow/DC/DF_OBS/1.0.0/*"
)


def _headers(accept=None):
    if not DC_API_KEY:
        raise ValueError("Set DC_API_KEY before requesting Data Commons.")
    headers = {"X-Api-Key": DC_API_KEY}
    if accept:
        headers["Accept"] = accept
    return headers


def sdmx_data(constraints):
    response = requests.get(
        SDMX_DATA_URL,
        params=constraints,
        headers=_headers("application/vnd.sdmx.data+csv;version=2.0.0"),
        timeout=120,
    )
    response.raise_for_status()
    frame = pd.read_csv(io.BytesIO(response.content), low_memory=False)
    frame["OBS_VALUE"] = pd.to_numeric(frame["OBS_VALUE"], errors="coerce")
    return frame


def availability_values(component, constraints):
    response = requests.get(
        f"{SDMX_AVAILABILITY_URL}/{component}",
        params=constraints,
        headers=_headers(),
        timeout=120,
    )
    response.raise_for_status()
    message = read_sdmx(io.BytesIO(response.content))
    if not message.structures:
        raise ValueError("The SDMX Availability request returned no constraint.")
    return {
        value.value
        for region in message.structures[0].cube_regions
        for key_value in region.key_values
        if key_value.id == component
        for value in key_value.values
    }


def require_variables(label, variables, constraints):
    constraints = {
        key: value for key, value in constraints.items()
        if not (key == "c[TIME_PERIOD]" and value == "LATEST")
    }
    available = availability_values(
        "variableMeasured",
        {**constraints, "c[variableMeasured]": ",".join(variables)},
    )
    missing = sorted(set(variables) - available)
    if missing:
        raise ValueError(f"{label} is missing variables: {', '.join(missing)}")
    print(f"{label}: all {len(variables)} variables are available.")


# The client follows response pagination within each input batch.
def _fetch_name_batch(nodes):
    return {
        dcid: name.value
        for dcid, name in dc_client.node.fetch_entity_names(
            entity_dcids=nodes
        ).items()
    }


def fetch_node_names(dcids, batch_size=500):
    unique_dcids = list(dict.fromkeys(dcids))
    if not unique_dcids:
        return {}
    batches = [
        unique_dcids[index:index + batch_size]
        for index in range(0, len(unique_dcids), batch_size)
    ]
    names = {dcid: dcid for dcid in unique_dcids}
    with ThreadPoolExecutor(max_workers=min(8, len(batches))) as executor:
        for batch_names in executor.map(_fetch_name_batch, batches):
            names.update(batch_names)
    return names


## Example: Median age vs. population by state, county, and city

For this exercise, we will compare median ages and population counts for US states, counties, and cities.


### Querying administrative areas with SDMX

The DCID for the United States is `country/USA`. The SDMX constraint `observationAbout.containedInPlace+` selects places directly or transitively contained in the United States. The `observationAbout.typeOf` constraint limits each request to states, counties, or cities.

The SDMX responses contain place DCIDs. The Data Commons Python client supplies their human-readable names.


### Querying statistics from Data Commons

Data Commons has a large corpus of statistical data, which can be queried and joined with other statistics. For example, we can query the median income of women living in Berkeley, California or the number of individuals who are insured in Maryland.

Before we explore how to do this, we need to understand how Data Commons stores statistical data. In particular, there are two types of entities: [StatisticalVariable](https://datacommons.org/browser/StatisticalVariable) and [StatVarObservation](https://datacommons.org/browser/StatVarObservation).

A StatisticalVariable represents any type of statistical metric that can be measured at a place and time. Some examples include median income, median income of females, number of high school graduates, unemployment rate, and prevalence of diabetes—essentially anything you might call a metric, statistic, or measure. A StatVarObservation represents an actual measurement of a StatisticalVariable in a given place and time.

One example of a StatVarObservation is the median age of people in San Antonio, Texas in 2024. The statistical metric, time, and place are median age, 2024, and San Antonio. You can explore statistical variables in the [Statistical Variable Explorer](https://datacommons.org/tools/statvar).

We are going to retrieve the total count and median age of populations—[Count_Person](https://datacommons.org/browser/Count_Person) and [Median_Age_Person](https://datacommons.org/browser/Median_Age_Person)—for US states, counties, and cities. Availability first confirms that both variables exist under the selected Census ACS 5-year facet. SDMX Data then returns their latest observations.

**Note:** The nationwide city query may take a minute.


In [ ]:
VARIABLES = ["Count_Person", "Median_Age_Person"]
PROVENANCE = "dc/base/CensusACS5YearSurvey"
MEASUREMENT_METHOD = "CensusACS5yrSurvey"


def fetch_geography(geography_type):
    constraints = {
        "c[observationAbout.containedInPlace+]": "country/USA",
        "c[observationAbout.typeOf]": geography_type,
        "c[provenance]": PROVENANCE,
        "c[measurementMethod]": MEASUREMENT_METHOD,
        "c[TIME_PERIOD]": "LATEST",
    }
    require_variables(geography_type, VARIABLES, constraints)
    rows = sdmx_data({
        **constraints,
        "c[variableMeasured]": ",".join(VARIABLES),
    }).dropna(subset=["OBS_VALUE"])

    complete = (
        rows.groupby("observationAbout")["variableMeasured"].nunique()
        == len(VARIABLES)
    )
    rows = rows[rows["observationAbout"].isin(complete[complete].index)]
    names = fetch_node_names(rows["observationAbout"].unique())
    return (
        rows.rename(columns={
            "observationAbout": "entity",
            "variableMeasured": "variable",
            "OBS_VALUE": "value",
        })
        .assign(entity_name=lambda frame: frame["entity"].map(names))
    )


df_state = fetch_geography("State")
df_county = fetch_geography("County")
df_city = fetch_geography("City")

print("states:", df_state["entity"].nunique())
print("counties:", df_county["entity"].nunique())
print("cities:", df_city["entity"].nunique())
df_city.head(5)


We can view the data we have queried.


### Cleaning and plotting the data

Great! It looks like we have all the data we need. Before we finish, let's do some post-processing.

We're interested in just the statistical values, so we'll select a single data point for each place-statistical-variable pair. Like the original tutorial, the pivot uses the human-readable place name as its row index, so counties or cities that share a name collapse into one row.


In [ ]:
def filter_to_stats_only(frame):
    frame = frame.pivot_table(
        index="entity_name",
        columns="variable",
        values="value",
        aggfunc="first",
    )
    frame = frame.rename_axis(None, axis=1)
    frame.index.name = None
    return frame


df_state = filter_to_stats_only(df_state)
df_county = filter_to_stats_only(df_county)
df_city = filter_to_stats_only(df_city)
df_city = df_city[df_city["Median_Age_Person"] >= 1]
df_city.head(5)


Finally, let's visualize our results.


In [ ]:
def plot_data(title, table):
    """Generate a scatter plot comparing median age and population."""
    plt.figure(figsize=(12, 8))
    plt.title(title)
    plt.xlabel("Median Age in Years")
    plt.ylabel("Population Count (log scale)")
    axis = plt.gca()
    axis.set_yscale("log")
    axis.scatter(
        table["Median_Age_Person"],
        table["Count_Person"],
        alpha=0.7,
    )


plot_data("Median Age vs. Population Count for States", df_state)
plot_data("Median Age vs. Population Count for Counties", df_county)
plot_data("Median Age vs. Population Count for Cities", df_city)


We can also plot each administrative-area granularity on the same plot to see how they relate.


In [ ]:
def plot_all_data(state_table, county_table, city_table):
    plt.figure(figsize=(12, 8))
    plt.title("Median Age vs. Population Count")
    plt.xlabel("Median Age in Years")
    plt.ylabel("Population Count (log scale)")

    state_color = "#ffa600"
    county_color = "#bc5090"
    city_color = "#003f5c"

    axis = plt.gca()
    axis.set_yscale("log")
    axis.scatter(
        state_table["Median_Age_Person"],
        state_table["Count_Person"],
        color=state_color,
        alpha=0.75,
    )
    axis.scatter(
        county_table["Median_Age_Person"],
        county_table["Count_Person"],
        color=county_color,
        alpha=0.5,
    )
    axis.scatter(
        city_table["Median_Age_Person"],
        city_table["Count_Person"],
        color=city_color,
        alpha=0.4,
    )

    plt.legend(handles=[
        mpatches.Patch(color=state_color, label="States"),
        mpatches.Patch(color=county_color, label="Counties"),
        mpatches.Patch(color=city_color, label="Cities"),
    ])


plot_all_data(df_state, df_county, df_city)


## What's next

Congratulations—you've completed the Data Commons task using SDMX. You can now explore other Data Commons notebooks for ideas about querying and joining data from the graph.
